# 01. 동공 시간곡선 (PsPM-AOB_UW) — 규칙 vs ML

**하는 일**: 밝기가 갑자기 바뀐 뒤 **첫 5초** 동안 동공이 어떻게 커지고 작아지는지를 공식(규칙)과 ML로 맞혀 보고, 같은 자로 채점해요.
그다음 이긴 쪽(또는 비슷하면 공식)으로 **경계부 끝(80m ≈ 2.9초)에서 동공이 얼마나 따라왔나(열림 %)**를 구해서 02 노트북에 넘겨요.

**왜 따로 하나**: 바깥 밝기(71786 사진 모델, 07 노트북)와 동공은 서로 다른 데이터로 만들어요. 07은 밝기만 내고, 여기는 그 밝기를 받아 동공을 계산해요. 둘을 잇는 건 02 노트북이에요.

**데이터**: PsPM-AOB_UW (Zenodo 8239465, CC BY). 500Hz 동공 지름(mm), 22명(전원 여성, 36~49세). 화면 회색(+) 5초 ↔ 원 5초(검정 0 · 0.25 · 0.75 · 흰 1).
- 화면 밝기(lx·cd/m²)는 원 자료에 없어요 → 여기서는 **바뀌는 모양(시간)**만 쓰고, **얼마나 바뀌나(크기)**는 02에서 SpectroSense로 검증한 W&Y 공식으로 정해요.
- SpectroSense는 10초 간격이라 시간 주장에 쓰지 않아요 (정상 상태 크기만).

**정답(y)**: 바뀐 순간 직전 0.2초 평균 대비 동공 지름 변화 ΔD(t), mm, 0.1~5.0초 0.1초 간격.

**채점**: 사람 단위 5겹 × 5번 (새 사람에게 맞나). 이김 기준 = MAE 0.1mm 이상 좋음 & 사람 다시 뽑기(bootstrap) 95% 하한 > 0 & 순위상관 같은 방향.

In [ ]:
import numpy as np, pandas as pd, scipy.io as sio, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
HERE = Path.cwd()
ROOT = next(p for p in [HERE, *HERE.parents] if (p / 'sunsight_data').exists())   # 머신러닝3조 폴더
DATA = ROOT / 'sunsight_data' / 'data' / 'Data'          # PsPM .mat 파일
OUT = ROOT / 'sunsight_pupil' / '결과'; OUT.mkdir(parents=True, exist_ok=True)
print('데이터:', DATA, '| .mat', len(list(DATA.glob('*.mat'))), '개')

## 1. 사건 뽑기
- 세션2(밝기 과제)만 써요. 표시(marker) = 화면이 바뀐 순간.
- 왼눈·오른눈 평균, 1.5mm 미만·9mm 초과(깜빡임)는 빈칸 → 앞뒤로 메움. 빈칸이 25% 넘는 사건은 버려요.
- 0.1초 칸으로 평균내서 50개 숫자(0.1~5.0초)로 만들어요.

In [ ]:
import numpy as np, pandas as pd, scipy.io as sio
from pathlib import Path

SR, BIN = 500, 0.1                       # 500Hz → 0.1초 칸으로 평균
T = np.round(np.arange(0.1, 5.0 + 1e-9, BIN), 1)   # 바뀐 뒤 0.1~5.0초
GRAY = 9
rows = []; info = []
for s in range(1, 24):
    fc, fp = DATA / f'AOB_UW_cogent_{s:02d}_sn2.mat', DATA / f'AOB_UW_pupil_{s:02d}_sn2.mat'
    if not (fc.exists() and fp.exists()):
        info.append((s, None, '파일 없음')); continue
    c = sio.loadmat(fc, squeeze_me=True, struct_as_record=False)
    p = sio.loadmat(fp, squeeze_me=True, struct_as_record=False)
    stim = c['data'][:, 1]; age = int(c['subject'].age)
    ch = {x.header.chantype: x for x in p['data']}
    pl, pr = np.asarray(ch['pupil_l'].data, float), np.asarray(ch['pupil_r'].data, float)
    pl[(pl < 1.5) | (pl > 9)] = np.nan; pr[(pr < 1.5) | (pr > 9)] = np.nan   # 눈 깜빡임·잡음 빼기
    pu = np.nanmean(np.vstack([pl, pr]), axis=0)
    mk = np.atleast_1d(ch['marker'].data)
    info.append((s, age, 'ok'))
    for j in range(1, len(mk)):
        on = int(round(mk[j] * SR))
        base = pu[on - int(0.2 * SR): on]
        seg = pu[on: on + int(5.0 * SR)]
        if len(seg) < int(5.0 * SR) or np.isnan(base).mean() > 0.5 or np.isnan(seg).mean() > 0.25:
            continue
        b = np.nanmean(base)
        binned = np.array([np.nanmean(seg[int((t - BIN) * SR): int(t * SR)]) for t in T])
        binned = pd.Series(binned).interpolate(limit_direction='both').values
        pre = pu[on - int(0.5 * SR): on]
        rows.append(dict(person=s, age=age, trial=j, before=stim[j - 1], after=stim[j], D0=b,
                         pre_slope=(np.nanmean(pre[-100:]) - np.nanmean(pre[:100])) / 0.3,
                         **{f't{t:.1f}': v - b for t, v in zip(T, binned)}))
ev = pd.DataFrame(rows)
def kind(r):
    if r.before == GRAY and r.after != GRAY: return f'회색→{r.after:g}'
    if r.after == GRAY and r.before != GRAY: return f'{r.before:g}→회색'
    return 'etc'
ev['전환'] = ev.apply(kind, axis=1)
print(pd.DataFrame(info, columns=['person', 'age', '상태']).query("상태!='ok'"))
print('사람', ev.person.nunique(), '명 · 전환', len(ev), '번'); print(ev.전환.value_counts())


## 2. 규칙 · ML · 채점
- **규칙A 평균곡선**: 학습 사람들의 전환별 평균 곡선을 그대로 씀 (가장 단순한 표)
- **규칙B 2단 공식**: ΔD(t) = A1·(1 − e^(−(t−d)/τ1)) + b·(t−d) — 지연 d초 뒤 빠른 단계(지수) + 느린 단계(5초 안에선 거의 직선). 숫자 4개만 학습 사람들로 맞춤
- **ML(전환·시각만)**: HistGradientBoosting, 입력 = 시각 t, 바뀌기 전·후 화면 값, 차이
- **ML(+동공크기·나이·직전추세)**: 위 + 바뀌기 전 동공 크기 D0, 나이, 직전 0.5초 추세 — 사람마다 다른 점을 ML이 잡나 보기

In [ ]:
from scipy.optimize import curve_fit
from scipy.stats import spearmanr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold
TC = [f't{t:.1f}' for t in T]
def two_stage(t, d, A1, tau1, b):           # 지연 d초 뒤 빠른 단계(지수) + 느린 단계(5초 안에선 거의 직선)
    x = np.clip(t - d, 0, None)
    return A1 * (1 - np.exp(-x / tau1)) + b * x
def fit_curve(curve):
    A0 = curve[-1]
    p0 = [0.3, A0, 0.8, 0.0]
    lo = [0.0, -6, 0.05, -1]; hi = [1.0, 6, 5.0, 1]
    p, _ = curve_fit(two_stage, T, curve, p0=p0, bounds=(lo, hi), maxfev=20000)
    return p
ev2 = ev[ev.전환 != 'etc'].reset_index(drop=True)
# 긴 형태(사건 × 시각) — ML 학습용
lv = lambda v: np.where(v == GRAY, 0.5, v)    # 회색 바탕은 0.5로 가정 (원문: 회색 + 표시)
long = ev2.melt(id_vars=['person', 'age', 'trial', 'before', 'after', 'D0', 'pre_slope', '전환'],
                value_vars=TC, var_name='tc', value_name='dD')
long['t'] = long.tc.str[1:].astype(float)
long['lv_before'], long['lv_after'] = lv(long.before), lv(long.after)
long['step'] = long.lv_after - long.lv_before
FEAT = ['t', 'lv_before', 'lv_after', 'step', 'D0', 'age', 'pre_slope']
FEAT_noD = ['t', 'lv_before', 'lv_after', 'step']

def cv_run(n_rep=5, seed=0):
    out = []
    persons = ev2.person.unique()
    rng = np.random.default_rng(seed)
    for rep in range(n_rep):
        perm = rng.permutation(persons); fold_of = {p: i % 5 for i, p in enumerate(perm)}
        fold = long.person.map(fold_of).values
        pred = {k: np.full(len(long), np.nan) for k in ['규칙A 평균곡선', '규칙B 2단 공식', 'ML(전환·시각만)', 'ML(+동공크기·나이·직전추세)']}
        for f in range(5):
            tr, te = fold != f, fold == f
            Ltr = long[tr]
            # 규칙 A: 학습 사람들의 전환별 평균 곡선
            mean_c = Ltr.groupby(['전환', 't']).dD.mean()
            pred['규칙A 평균곡선'][te] = mean_c.reindex(pd.MultiIndex.from_arrays([long.전환[te], long.t[te]])).values
            # 규칙 B: 평균 곡선에 2단 공식(지연·빠른 τ·느린 기울기) 맞춘 값
            pB = {}
            for k, g in mean_c.groupby(level=0):
                pB[k] = fit_curve(g.values)
            pred['규칙B 2단 공식'][te] = [two_stage(t, *pB[k]) for k, t in zip(long.전환[te], long.t[te])]
            for name, cols in [('ML(전환·시각만)', FEAT_noD), ('ML(+동공크기·나이·직전추세)', FEAT)]:
                m = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, max_leaf_nodes=15,
                                                  min_samples_leaf=40, random_state=rep)
                m.fit(Ltr[cols], Ltr.dD)
                pred[name][te] = m.predict(long.loc[te, cols])
        for k, v in pred.items():
            out.append(pd.DataFrame({'rep': rep, '모델': k, 'person': long.person, '전환': long.전환,
                                     't': long.t, 'trial': long.trial, 'y': long.dD, 'p': v}))
    return pd.concat(out, ignore_index=True)
oof = cv_run()
oof['ae'] = (oof.y - oof.p).abs()
def summary(o):
    r = {}
    r['MAE 전체(mm)'] = o.ae.mean()
    r['MAE 1초(mm)'] = o[np.isclose(o.t, 1.0)].ae.mean()
    r['MAE 3초(mm)'] = o[np.isclose(o.t, 3.0)].ae.mean()
    s3 = o[np.isclose(o.t, 3.0)]
    r['순위상관 3초'] = spearmanr(s3.y, s3.p).correlation
    return pd.Series(r)
tab = oof.groupby('모델').apply(summary).round(3)
print(tab)
key = oof[oof.전환.isin(['회색→0', '회색→1'])]
print(key.groupby(['전환', '모델']).apply(summary).round(3))


### 짝지은 차이 (규칙B − ML, 사람 단위)
양수면 ML이 더 잘 맞힘. **이김** = 차이 ≥ 0.1mm & 95% 하한 > 0.

In [ ]:
def paired(o, ml, rule='규칙B 2단 공식', n_boot=2000, seed=1):
    pm = o[o.모델.isin([ml, rule])].groupby(['모델', 'person']).ae.mean().unstack(0)
    diff = (pm[rule] - pm[ml]).values                     # 규칙 − ML (양수 = ML이 나음)
    rng = np.random.default_rng(seed)
    bs = [rng.choice(diff, len(diff)).mean() for _ in range(n_boot)]
    lo, hi = np.percentile(bs, [2.5, 97.5])
    return dict(비교=f'{rule} − {ml}', 차이_mm=diff.mean(), 하한=lo, 상한=hi, 사람수=len(diff))
cmp = []
for sub, o in [('전체', oof), ('회색→0 (어두워짐 = 입구)', oof[oof.전환 == '회색→0']), ('회색→1 (밝아짐 = 출구)', oof[oof.전환 == '회색→1'])]:
    for ml in ['ML(+동공크기·나이·직전추세)', 'ML(전환·시각만)']:
        cmp.append(dict(범위=sub, **paired(o, ml)))
cmp = pd.DataFrame(cmp)
cmp['판정'] = np.where((cmp.차이_mm >= 0.1) & (cmp.하한 > 0), 'ML 이김',
              np.where(cmp.하한 > 0, 'ML 조금 나음 (0.1mm 못 넘음)', '비슷 / ML 못 이김'))
tab.to_csv(OUT / '동공시간_비교표.csv', encoding='utf-8-sig')
cmp.to_csv(OUT / '동공시간_짝차이.csv', index=False, encoding='utf-8-sig')
display(tab); display(cmp.round(3))

## 3. 공식 숫자 (전체 22명 평균 곡선) + 사람별 퍼짐
- d = 반응 시작까지 걸리는 시간(초), τ1 = 빠른 단계 시간(초), A1 = 빠른 단계 크기(mm), b = 5초 안 느린 단계 기울기(mm/초)
- 어두워질 때(회색→0): 천천히 커지고 5초에도 계속 커짐 (느린 단계)
- 밝아질 때(회색→1): 0.3초 뒤 빠르게 줄고(τ1 짧음), 그 뒤 살짝 다시 커짐(b > 0, '동공 탈출' 현상)

In [ ]:
# 전체 사람 평균 곡선에 2단 공식 + 사람별 공식 (퍼짐 보기)
par = []
for k in ['회색→0', '회색→1', '회색→0.25', '회색→0.75']:
    g = ev2[ev2.전환 == k]
    p = fit_curve(g[TC].mean().values); par.append(dict(전환=k, 대상='전체 평균', d=p[0], A1=p[1], tau1=p[2], b=p[3]))
    for s, gs in g.groupby('person'):
        try:
            p = fit_curve(gs[TC].mean().values); par.append(dict(전환=k, 대상=f'p{s:02d}', d=p[0], A1=p[1], tau1=p[2], b=p[3]))
        except Exception: pass
par = pd.DataFrame(par)
print(par[par.대상 == '전체 평균'].round(3))
print(par[par.대상 != '전체 평균'].groupby('전환')[['d', 'A1', 'tau1', 'b']].quantile([.25, .5, .75]).round(3))
par.to_csv(OUT / '동공시간_공식숫자.csv', index=False, encoding='utf-8-sig')

## 4. 정상 상태 대비 얼마나 따라왔나 (열림 %)
- 5초 자료로는 **끝까지 다 바뀐 크기**를 모름 → 느린 단계를 τ2초짜리 지수로 보고 τ2 = 10 · 30 · 60초 가정 (문헌 평형 10~120초, Zandi 2021) → **민감도**로 보여줌
- 느린 단계 크기 A2 = b × τ2 (처음 기울기가 b가 되도록)
- **상한**: 5초 값을 끝으로 본다면 2.9초 값은 그 몇 %인지도 같이 (실제는 이보다 낮음)
- 밝아질 때(출구)는 0.5초 안에 빠른 단계가 거의 끝나고 그 뒤는 되레 커지므로 **1초 이후 100%로 봄** (줄어든 뒤에도 남는 밝기 차이는 망막이 감당)

In [ ]:
def frac_dark(t, d, A1, tau1, b, tau2):
    """어두워질 때: 끝까지 다 커진 크기 대비 t초에 커진 비율"""
    x = np.clip(np.asarray(t, float) - d, 0, None); A2 = b * tau2
    return (A1 * (1 - np.exp(-x / tau1)) + A2 * (1 - np.exp(-x / tau2))) / (A1 + A2)
def frac_bright(t, d, A1, tau1, b):
    """밝아질 때: 빠른 단계 크기 대비 t초에 줄어든 비율 (느린 되커짐은 뺌)"""
    x = np.clip(np.asarray(t, float) - d, 0, None); return 1 - np.exp(-x / tau1)
V_KMH, ZONE_M = 100, 80                                     # 설계속도, 경계부 길이(국토부 지침 그림 6.4)
T_ZONE = ZONE_M / (V_KMH / 3.6)
P = par[par.대상 == '전체 평균'].set_index('전환')
mean5 = ev2.groupby('전환')[TC].mean()
rows = []
for tau2 in [10, 30, 60]:
    r = P.loc['회색→0']
    rows.append(dict(방향='입구 (어두워짐)', 가정=f'τ2={tau2}초', **{f'{t:g}초': frac_dark(t, r.d, r.A1, r.tau1, r.b, tau2) for t in [1, 2, T_ZONE, 5]}))
rows.append(dict(방향='입구 (어두워짐)', 가정='상한: 5초 값을 끝으로', **{f'{t:g}초': np.interp(t, T, mean5.loc['회색→0'].values) / mean5.loc['회색→0', 't5.0'] for t in [1, 2, T_ZONE, 5]}))
r = P.loc['회색→1']
rows.append(dict(방향='출구 (밝아짐)', 가정='빠른 단계 기준', **{f'{t:g}초': frac_bright(t, r.d, r.A1, r.tau1, r.b) for t in [1, 2, T_ZONE, 5]}))
open_tab = pd.DataFrame(rows).rename(columns={f'{T_ZONE:g}초': f'경계부끝 {T_ZONE:.1f}초'})
open_tab.to_csv(OUT / '동공시간_열림비율.csv', index=False, encoding='utf-8-sig')
print(f'경계부 {ZONE_M}m ÷ {V_KMH}km/h = {T_ZONE:.2f}초'); display(open_tab.round(2))

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'DejaVu Sans']; plt.rcParams['axes.unicode_minus'] = False
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, k, title in [(ax[0], '회색→0', '어두워질 때 (입구와 같은 방향)'), (ax[1], '회색→1', '밝아질 때 (출구와 같은 방향)')]:
    g = ev2[ev2.전환 == k]
    for s, gs in g.groupby('person'): a.plot(T, gs[TC].mean().values, color='0.8', lw=0.7)
    a.plot(T, g[TC].mean().values, 'k', lw=2, label='22명 평균')
    a.plot(T, two_stage(T, *P.loc[k, ['d', 'A1', 'tau1', 'b']].values), 'r--', lw=2, label='2단 공식')
    a.axvline(T_ZONE, color='b', ls=':', label=f'경계부 끝 {T_ZONE:.1f}초')
    a.set_title(title); a.set_xlabel('바뀐 뒤 시간 (초)'); a.set_ylabel('동공 변화 ΔD (mm)'); a.legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT / '동공시간_곡선.png', dpi=130); plt.show()

## 5. 읽는 법 · 한계
- **규칙 vs ML**: 2단 공식이 ML과 비슷하면(위 짝차이 표) → **공식 사용**. 동공 시간 반응은 숫자 4개짜리 공식으로 충분히 설명된다는 뜻.
- **넘기는 값**: `결과/동공시간_공식숫자.csv`(전체 평균 줄) → 02 노트북이 읽어 경계부 끝 동공 크기를 계산.
- **한계**: 22명 전원 여성 36~49세 (나이·성별 일반화 X), 화면 밝기 단위 없음(크기는 W&Y 공식으로), 실험실 화면 = 실제 터널 아님, 5초 이후 느린 단계는 τ2 가정.